# Annotate Research Papers with Text Embeddings

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](<colab_link>)

> **Google Colab:** If imports fail after installation, select
> **Runtime → Restart session**, then run the notebook again.

Map queried text embeddings back to arXiv papers and collect labels from
their titles and abstracts. A logistic regression classifier fits acquired
labels, and an independent classifier fits reference labels for the same
queried papers. The published run selects `AUTOMATIC = True`, which submits
category-derived reference answers through the shared workflow. Set it to
`False` to provide your own judgments. This example covers widget callbacks,
annotation sessions, interruption recovery, results, and export. Read the
[pool-based introduction](00_pool_getting_started.ipynb) for the query loop.

## Setup

Uncomment the next cell if dependencies are missing. The first run downloads
papers and their embeddings from Hugging Face into `CACHE_DIR`. Later runs
reuse that cache. Manual annotation needs a live Jupyter notebook
with widget support. Buttons in the saved documentation cannot submit labels
without a running kernel.

In [1]:
# !pip install scikit-activeml datasets ipywidgets jupyter_ui_poll matplotlib

Select the dataset size and annotation mode before loading data. The
editable `MAX_PER_CLASS = 20` cap chooses at most 20 papers per eligible
reference class before converting embeddings to NumPy. Set it to `None`
to use as many papers per class as the smaller eligible group contains.
The classes, 75/25 stratified split, seed, and batches of five stay the same.
`AUTOMATIC = True` produces the published reference-label demonstration.
For manual annotation, set `AUTOMATIC = False` and rerun the notebook.
`SHOW_REFERENCE` separately controls whether the paper view reveals its
category-derived answer.

In [2]:
AUTOMATIC = True
SHOW_REFERENCE = False
MAX_PER_CLASS = 20
RANDOM_STATE = 0
BATCH_SIZE = 5
CACHE_DIR = "huggingface_cache/datasets/"

Import dataset access, widget callbacks, and the classifier and query
strategy. Each session owns its labels and controls, so creating another
session starts with fresh state.

In [3]:
import logging
import warnings

logging.disable(logging.WARNING)
warnings.filterwarnings("ignore")

import sys
import textwrap
import time

import datasets
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from jupyter_ui_poll import ui_events
from tqdm import tqdm
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

from skactiveml.classifier import SklearnClassifier
from skactiveml.pool import RandomSampling
from skactiveml.utils import (
    MISSING_LABEL,
    call_func,
    is_labeled,
    unlabeled_indices,
)

plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.labelsize": 11,
        "axes.titlesize": 12,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
        "legend.fontsize": 10,
        "figure.titlesize": 12,
    }
)

Use the [MongoDB arXiv dataset](https://huggingface.co/datasets/MongoDB/subset_arxiv_papers_with_embeddings)
to define a binary task from paper categories:

- **Machine learning (1):** `cs.AI`, `cs.LG`, or `cs.CV`.
- **Other (0):** `cs.LO`, `cs.NI`, `cs.CR`, or `cs.PL`.

A paper is eligible only if its categories intersect exactly one group.
For example, `cs.LG cs.AI` belongs to class 1, while `cs.LG cs.CR` is
excluded. These category-derived labels are a proxy for the reading task.
Disagreement with a reader's judgment is not automatically an annotation
error.

## Prepare a Balanced Pool

Use seed 0 to select equal numbers from the two eligible reference classes.
Keep original source indices beside the embeddings so each query maps
back to a title and abstract. Reference labels remain separate from acquired
labels. The cap limits selected rows before embedding conversion, although
the dataset itself may still be downloaded and cached in full.

In [4]:
rng = np.random.RandomState(RANDOM_STATE)
papers = datasets.load_dataset(
    "MongoDB/subset_arxiv_papers_with_embeddings",
    cache_dir=CACHE_DIR,
    split="train",
)

other_categories = {"cs.LO", "cs.NI", "cs.CR", "cs.PL"}
ml_categories = {"cs.AI", "cs.LG", "cs.CV"}
category_sets = [set(value.split()) for value in papers["categories"]]
is_other = np.array([bool(c & other_categories) for c in category_sets])
is_ml = np.array([bool(c & ml_categories) for c in category_sets])
other_indices = np.flatnonzero(is_other & ~is_ml)
ml_indices = np.flatnonzero(is_ml & ~is_other)

samples_per_class = min(len(other_indices), len(ml_indices))
if MAX_PER_CLASS is not None:
    samples_per_class = min(samples_per_class, MAX_PER_CLASS)
paper_indices = np.concatenate(
    [
        rng.choice(other_indices, samples_per_class, replace=False),
        rng.choice(ml_indices, samples_per_class, replace=False),
    ]
)
X_embeddings = np.asarray(
    papers.select(paper_indices.tolist())["embedding"], dtype=float
)
y_reference = np.repeat([0.0, 1.0], samples_per_class)

Hold out 25% with a stratified split and seed 0. With 20 papers per class,
this gives 30 pool papers and 10 held-out papers. Declare the vocabulary
`[0, 1]` before acquisition. The query call receives pool embeddings and
acquired labels. Held-out reference labels are used only for evaluation.

In [5]:
(
    paper_indices_train,
    paper_indices_test,
    X_train,
    X_test,
    y_train_reference,
    y_test,
) = train_test_split(
    paper_indices,
    X_embeddings,
    y_reference,
    test_size=0.25,
    stratify=y_reference,
    random_state=RANDOM_STATE,
)

| Object | Role |
| --- | --- |
| `papers` | Source records with titles, abstracts, categories, and embeddings. |
| `paper_indices_train` | Maps each pool row to its source record. |
| `X_train`, `X_test` | Fixed vectors for fitting/querying and held-out evaluation. |
| `y_train_reference`, `y_test` | Category-derived targets for comparison and evaluation. |
| Session's `y_train` | Initially missing labels, filled by annotation responses. |

For query index 3, the displayed record is
`papers[int(paper_indices_train[3])]` and the response fills position 3
of the session label vector. Class 0 is an observed answer, while
`MISSING_LABEL` marks an unanswered paper.

## Build the Annotation Interface

The view displays a title and abstract. **Machine learning** submits 1,
**Other** submits 0, and **Use reference** submits the category-derived
answer. `SHOW_REFERENCE = True` also displays that answer beside the paper.
`PaperAnnotationUI` enables controls while waiting and disables them after
the first accepted click. A second click cannot overwrite the response.
`jupyter_ui_poll` processes button events while Python waits. Automatic
mode returns the reference answer without waiting. There is no skip or
undo here. The [image tutorial](03_pool_oracle_annotations.ipynb) implements
those controls.

Define this support code before starting a session. `normalize_text` wraps
paper text. The UI handles one response at a time, updates checkpoint
scores and curves, and disables controls on completion or interruption.

In [6]:
def normalize_text(text):
    return textwrap.fill(" ".join(text.split()), width=70)


class PaperAnnotationUI:
    """Collect one label per paper and display batch-level evaluation."""

    def __init__(
        self,
        papers,
        paper_indices,
        reference_labels,
        show_reference=False,
        automatic=False,
    ):
        self.papers = papers
        self.paper_indices = paper_indices
        self.reference_labels = reference_labels
        self.show_reference = show_reference
        self.automatic = automatic
        self._waiting = False
        self._response = None
        self.stopped = False
        self.output_sample = widgets.Output()
        self.output_stats = widgets.Output()
        self.output_plot = widgets.Output()
        self.buttons = []
        for title, label in [
            ("Machine learning", 1),
            ("Other", 0),
            ("Use reference", -1),
        ]:
            button = widgets.Button(description=title, disabled=True)
            button.on_click(lambda _, value=label: self._submit(value))
            self.buttons.append(button)
        self.widget = widgets.VBox(
            [
                self.output_stats,
                self.output_plot,
                widgets.HBox([widgets.VBox(self.buttons), self.output_sample]),
            ]
        )

    def _submit(self, label):
        if self._waiting:
            self._response = label
            self.stop_waiting()

    def stop_waiting(self):
        self._waiting = False
        for button in self.buttons:
            button.disabled = True

    def annotate(self, query_indices):
        labels = []
        for position, query_index in enumerate(query_indices, start=1):
            paper = self.papers[int(self.paper_indices[query_index])]
            with self.output_sample:
                self.output_sample.clear_output(wait=True)
                print(
                    f"Paper {position} of {len(query_indices)} in this batch"
                )
                if self.show_reference:
                    name = (
                        "Machine learning"
                        if self.reference_labels[query_index]
                        else "Other"
                    )
                    print(f"Reference category: {name}")
                print(f"\nTitle: {normalize_text(paper['title'])}")
                print(f"\nAbstract:\n{normalize_text(paper['abstract'])}")

            if self.automatic:
                response = -1
            else:
                self._response = None
                self._waiting = True
                for button in self.buttons:
                    button.disabled = False
                try:
                    with ui_events() as poll:
                        while self._waiting:
                            poll(10)
                            time.sleep(0.01)
                    response = self._response
                except KeyboardInterrupt:
                    self.stopped = True
                    break
                finally:
                    self.stop_waiting()
            labels.append(
                self.reference_labels[query_index]
                if response == -1
                else response
            )
        return np.asarray(labels, dtype=float)

    def update(self, counts, accuracies, reference_accuracies, disagreements):
        with self.output_stats:
            self.output_stats.clear_output(wait=True)
            print(f"Acquired labels: {counts[-1]}")
            if not self.automatic:
                print(
                    f"Disagreements with reference: {disagreements}/{counts[-1]}"
                )
        with self.output_plot:
            self.output_plot.clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(7.2, 3.4), layout="constrained")
            label = "Reference labels" if self.automatic else "Acquired labels"
            ax.plot(
                counts,
                accuracies,
                marker="o",
                markersize=4,
                color="#0072B2",
                label=f"{label} ({np.mean(accuracies):.3f})",
            )
            if not self.automatic:
                ax.plot(
                    counts,
                    reference_accuracies,
                    color="#D55E00",
                    linestyle="--",
                    label=f"Reference labels ({np.mean(reference_accuracies):.3f})",
                )
            ax.set(
                xlabel="Acquired labels",
                ylabel="Held-out reference accuracy",
            )
            ax.margins(x=0.03, y=0.08)
            ax.grid(axis="y", alpha=0.2)
            ax.legend(title="Mean accuracy", frameon=False)
            display(fig)
            plt.close(fig)

    def finish(self):
        self.stop_waiting()
        with self.output_sample:
            self.output_sample.clear_output(wait=True)
            if self.stopped:
                print(
                    "Session stopped. Acquired labels are available in session."
                )
            else:
                print("All papers in the training pool have been annotated.")

## Run an Annotation Session

Two independent logistic regression models fit exactly the same queried
papers. One uses acquired responses and the other uses their reference
labels. Both refit from scratch on all observed labels and score against
the same held-out references. Their separation shows the effect of
disagreements for this query sequence. The reference-label model is not a
separate acquisition run or a guaranteed upper bound on accuracy.

`RandomSampling` selects the papers. Replace the strategy or classifier
inside the function to try another query rule. `call_func` forwards only
arguments supported by the strategy. The query receives acquired labels,
with `fit_clf=False` because the model was fitted at the last checkpoint.
An adaptive replacement would use that model to choose subsequent papers.

Record the zero-label baseline, then repeat query → collect responses →
reveal the same papers' references to the comparison model → refit →
evaluate. With 30 pool papers, checkpoints contain 0, 5, 10, 15, 20, 25,
and 30 acquired labels. A smaller final batch is supported.

Accuracy is the fraction of held-out references predicted correctly. The
displayed mean averages checkpoints with equal weight, including the
baseline. It does not describe variability across data splits. In manual
mode, interrupt the kernel while waiting to stop early. Responses already
entered in that batch are retained and evaluated. For example, two
responses from a batch of five produce a two-label checkpoint. The returned
`session["stopped"]` is then `True`.

In [7]:
def run_annotation_session(
    X_train,
    y_train_reference,
    X_test,
    y_test,
    papers,
    paper_indices_train,
    batch_size=BATCH_SIZE,
    automatic=AUTOMATIC,
    show_reference=SHOW_REFERENCE,
):
    y_train = np.full(len(X_train), MISSING_LABEL)
    y_train_ideal = np.full(len(X_train), MISSING_LABEL)
    clf = SklearnClassifier(
        LogisticRegression(random_state=RANDOM_STATE),
        classes=[0, 1],
        random_state=RANDOM_STATE,
    )
    clf_ideal = clone(clf)
    qs = RandomSampling(random_state=RANDOM_STATE)
    ui = PaperAnnotationUI(
        papers,
        paper_indices_train,
        y_train_reference,
        show_reference=show_reference,
        automatic=automatic,
    )
    display(ui.widget)
    counts, accuracies, reference_accuracies = [], [], []
    disagreements = 0

    def record_scores():
        clf.fit(X_train, y_train)
        clf_ideal.fit(X_train, y_train_ideal)
        accuracy = clf.score(X_test, y_test)
        reference_accuracy = clf_ideal.score(X_test, y_test)
        counts.append(int(is_labeled(y_train).sum()))
        accuracies.append(accuracy)
        reference_accuracies.append(reference_accuracy)
        ui.update(counts, accuracies, reference_accuracies, disagreements)

    progress = tqdm(
        total=len(X_train), desc="Paper annotations", unit="paper",
        file=sys.stdout, ascii=True, mininterval=2.0, ncols=80, leave=True,
    )
    try:
        record_scores()
        while len(unlabeled_indices(y_train)):
            query_indices = call_func(
                qs.query,
                X=X_train,
                y=y_train,
                clf=clf,
                fit_clf=False,
                batch_size=min(batch_size, len(unlabeled_indices(y_train))),
            )
            labels = ui.annotate(query_indices)
            if not len(labels):
                break
            query_indices = query_indices[: len(labels)]
            y_train[query_indices] = labels
            y_train_ideal[query_indices] = y_train_reference[query_indices]
            disagreements += int(
                np.sum(labels != y_train_reference[query_indices])
            )
            progress.update(len(labels))
            record_scores()
            if ui.stopped:
                break
    except KeyboardInterrupt:
        ui.stopped = True
    finally:
        ui.stop_waiting()
        progress.close()
    ui.finish()

    return {
        "labels": y_train,
        "reference_labels": y_train_ideal,
        "annotation_counts": np.asarray(counts),
        "accuracy": np.asarray(accuracies),
        "reference_accuracy": np.asarray(reference_accuracies),
        "disagreements": disagreements,
        "stopped": ui.stopped,
        "automatic": automatic,
        "ui": ui,
    }

Start a fresh session below. The published automatic run annotates every
pool paper from references. In manual mode, use the buttons until completion
or interrupt while waiting. Rerunning creates new labels, models, controls,
and results. It does not resume the previous session, so export first if
you need its labels.

In [8]:
session = run_annotation_session(
    X_train, y_train_reference, X_test, y_test, papers, paper_indices_train
)

Paper annotations: 100%|#####################| 30/30 [00:00<00:00, 48.78paper/s]


The saved widget shows the completed automatic reference-label session.
Its scores and plot are the final checkpoint, and annotation buttons are
disabled. Manual callbacks require a live notebook kernel.

The widget updates after each batch. Legend values average the evaluated
checkpoints, including the initial model. Automatic mode trains both models
on identical reference labels, so it shows one curve. Manual mode also
shows the model trained on references for the same queried papers.

`session["labels"]` follows `X_train` row order, and `paper_indices_train`
maps those rows to source records. Missing entries identify unanswered papers.
The returned count and accuracy arrays retain each evaluated checkpoint,
including an interrupted partial batch.

## Export and Adapt the Session

Copy the labels and source indices together so every saved response can be
traced back to its paper. The next cell creates an export snapshot and
shows the first three mapped responses. Uncomment `np.savez` to persist it
after the kernel stops. In an interrupted session, missing entries remain
missing and the already acquired labels are included. Rerun this cell to
refresh the snapshot after another session.

In [9]:
paper_label_export = {
    "labels": session["labels"].copy(),
    "paper_indices": paper_indices_train.copy(),
}
print(f"Export: {is_labeled(paper_label_export['labels']).sum()} acquired labels.")
print("Pool row | Source record | Acquired label")
for pool_row in range(min(3, len(X_train))):
    print(
        f"{pool_row:8d} | "
        f"{int(paper_label_export['paper_indices'][pool_row]):13d} | "
        f"{paper_label_export['labels'][pool_row]}"
    )

# np.savez("paper_labels.npz", **paper_label_export)

Export: 30 acquired labels.
Pool row | Source record | Acquired label
       0 |         41728 | 1.0
       1 |         17425 | 0.0
       2 |          4971 | 1.0


To read a saved export, use `np.load("paper_labels.npz")` and retain
both arrays. The saved label vector records responses, while review
controls and checkpoint histories belong to the live session. Try
`AUTOMATIC = False` to inspect label disagreements, or replace
`RandomSampling` with `UncertaintySampling` to make later queries
depend on the acquired-label model. Increase `MAX_PER_CLASS` for a
larger reading task and assessment set. This single split and query
sequence does not establish a strategy ranking. The
[evaluation study](04_pool_simple_evaluation_study.ipynb) introduces
repeated comparisons, and the
[image tutorial](03_pool_oracle_annotations.ipynb) adds skip and undo.